<a href="https://colab.research.google.com/github/sairahul1526/pitch-deck-outline/blob/main/notebooks/ocr_benchmark_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pitch-deck OCR benchmark

This notebook is a provider-neutral Colab runner for the private OCR sample. It keeps raw PDFs and gold annotations in your Google Drive and writes only hashed benchmark artifacts. Run the native baseline first; enable Docling or PaddleOCR only after the runtime is ready.

## 1. Runtime and data paths

Use a GPU runtime only for the optional VLM cells. Upload either the local `data/raw` directory or the generated `pitch-deck-outline-raw.tar` archive into this Drive folder before running the benchmark. Do not upload permission emails or private gold transcriptions to GitHub.

In [ ]:
%pip install -q pypdf
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')
DATA_ROOT = Path('/content/drive/MyDrive/pitch-deck-outline-data')
RAW_ROOT = DATA_ROOT / 'raw'
RUN_ROOT = DATA_ROOT / 'runs' / 'ocr'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print('raw root:', RAW_ROOT)
print('exists:', RAW_ROOT.exists())

In [ ]:
import hashlib
import tarfile

archive_path = DATA_ROOT / 'pitch-deck-outline-raw.tar'
awesome_root = RAW_ROOT / 'awesome-pitch-decks' / 'pdfs'
if archive_path.exists() and not awesome_root.exists():
    with tarfile.open(archive_path, 'r') as archive:
        archive.extractall(DATA_ROOT, filter='data')

def unique_valid_pdfs(root: Path) -> list[Path]:
    unique: dict[str, Path] = {}
    for path in sorted(root.glob('*.pdf')):
        with path.open('rb') as handle:
            if handle.read(5) != b'%PDF-':
                continue
            digest = hashlib.sha256()
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                digest.update(chunk)
        unique.setdefault(digest.hexdigest(), path)
    return sorted(unique.values())

awesome_pdfs = unique_valid_pdfs(awesome_root)
pitch_deckz_root = RAW_ROOT / 'huggingface' / 'skyforclouds__pitch-deckz' / 'files'
pitch_deckz_pdfs = unique_valid_pdfs(pitch_deckz_root)
print('Awesome Pitch Decks (unique valid PDFs):', len(awesome_pdfs))
print('Pitch Deckz (unique valid PDFs):', len(pitch_deckz_pdfs))
print('all PDFs:', len(awesome_pdfs) + len(pitch_deckz_pdfs))
assert awesome_pdfs, 'Upload data/raw or pitch-deck-outline-raw.tar into the Drive folder first'

In [ ]:
# The previous cell performs PDF-header validation and content-hash de-duplication.
sample = awesome_pdfs[:5]
print('sample:', [path.name for path in sample])

## 2. Native PDF baseline

This is the speed baseline. It is intentionally conservative: pages with little extracted text should be routed to OCR rather than silently accepted.

In [ ]:
import json
import time

from pypdf import PdfReader


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def native_extract(path: Path) -> dict:
    started = time.perf_counter()
    reader = PdfReader(str(path))
    pages = []
    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ''
        pages.append({'page_number': page_number, 'text': text, 'characters': len(text)})
    return {
        'engine': 'native-pypdf',
        'engine_version': 'colab-runtime',
        'source_file': str(path),
        'input_sha256': sha256_file(path),
        'latency_ms': round((time.perf_counter() - started) * 1000, 2),
        'pages': pages,
    }

sample = awesome_pdfs[:5]
native_results = [native_extract(path) for path in sample]
print([(Path(item['source_file']).name, len(item['pages'])) for item in native_results])

In [ ]:
native_report = RUN_ROOT / 'native-smoke.json'
native_report.write_text(json.dumps(native_results, indent=2) + '\n')
print(native_report)

## 3. Docling adapter (recommended first OCR candidate)

Docling is the first layout-aware candidate. It may install additional model assets on first use; keep its cache on the runtime or a disposable Drive cache.

In [ ]:
%pip install -q docling
from docling.document_converter import DocumentConverter

converter = DocumentConverter()

def docling_extract(path: Path) -> dict:
    started = time.perf_counter()
    result = converter.convert(str(path))
    markdown = result.document.export_to_markdown()
    return {
        'engine': 'docling',
        'engine_version': 'colab-installed',
        'source_file': str(path),
        'input_sha256': sha256_file(path),
        'latency_ms': round((time.perf_counter() - started) * 1000, 2),
        'text': markdown,
    }

docling_result = docling_extract(sample[0])
print(docling_result['source_file'], len(docling_result['text']))

## 4. Expanded 50-page benchmark

The native baseline is measured page by page for exactly 50 pages. Docling is then run across the same five representative decks so layout-aware OCR output and latency can be reviewed together. This is a routing benchmark, not a quality claim until private gold transcriptions are reviewed.

In [ ]:
benchmark_decks = sample
native_page_rows = []
for path in benchmark_decks:
    reader = PdfReader(str(path))
    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ''
        native_page_rows.append({
            'source_file': path.name,
            'page_number': page_number,
            'characters': len(text),
            'empty': not bool(text.strip()),
        })

native_50_pages = native_page_rows[:50]
native_50_report = {
    'evidence_scope': 'expanded_50_page_sample',
    'engine': 'native-pypdf',
    'decks_considered': [path.name for path in benchmark_decks],
    'pages_sampled': len(native_50_pages),
    'characters': sum(row['characters'] for row in native_50_pages),
    'empty_pages': sum(row['empty'] for row in native_50_pages),
    'page_rows': native_50_pages,
}
native_50_path = RUN_ROOT / 'native-50-page-report.json'
native_50_path.write_text(json.dumps(native_50_report, indent=2) + '\n')

docling_runs = []
for path in benchmark_decks:
    result = docling_result if path == sample[0] else docling_extract(path)
    docling_runs.append({
        'source_file': path.name,
        'input_sha256': result['input_sha256'],
        'characters': len(result['text']),
        'latency_ms': result['latency_ms'],
        'native_pages': len(PdfReader(str(path)).pages),
    })
docling_5_report = {
    'evidence_scope': 'expanded_50_page_sample',
    'engine': 'docling',
    'documents_sampled': len(docling_runs),
    'source_pages_covered': sum(row['native_pages'] for row in docling_runs),
    'runs': docling_runs,
}
docling_5_path = RUN_ROOT / 'docling-5-deck-report.json'
docling_5_path.write_text(json.dumps(docling_5_report, indent=2) + '\n')

combined_report = {
    'evidence_scope': 'expanded_50_page_sample',
    'native': native_50_report,
    'docling': docling_5_report,
    'notes': (
        'Native metrics are page-level for exactly 50 pages. '
        'Docling runs cover the same five decks; compare quality after private review.'
    ),
}
combined_path = RUN_ROOT / 'ocr-expanded-report.json'
combined_path.write_text(json.dumps(combined_report, indent=2) + '\n')
print('native:', native_50_path)
print('docling:', docling_5_path)
print('combined:', combined_path)
print(
    'native pages:', native_50_report['pages_sampled'],
    'characters:', native_50_report['characters'],
    'empty:', native_50_report['empty_pages'],
)
print(
    'docling decks:', docling_5_report['documents_sampled'],
    'source pages:', docling_5_report['source_pages_covered'],
)

## 5. Optional PaddleOCR-VL adapter

Run this only after the native and Docling smoke checks pass. PaddleOCR-VL has a separate PaddleX OCR extra and the plain `paddlepaddle` wheel may be CPU-only even when Colab exposes a GPU. This cell records both conditions and writes a diagnostic artifact; a failed initialization is evidence about the runtime, not a quality score.

In [ ]:
import importlib.metadata
import json
import subprocess
import sys
import traceback

paddle_report = {
    'evidence_scope': 'single_deck_api_smoke',
    'engine': 'paddleocr-vl',
    'source_file': sample[0].name,
}

# PaddleOCR-VL requires the PaddleX OCR extra. Keep installation in the
# notebook so a fresh Colab runtime is reproducible.
install = subprocess.run(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        'paddleocr',
        'paddlepaddle',
        'paddlex[ocr]',
    ],
    capture_output=True,
    text=True,
    check=False,
)
paddle_report['pip_returncode'] = install.returncode
if install.returncode != 0:
    paddle_report['status'] = 'install_failed'
    paddle_report['pip_stderr_tail'] = install.stderr[-4000:]
else:
    try:
        import paddle
        from paddleocr import PaddleOCRVL

        paddle_report['paddle_version'] = getattr(paddle, '__version__', 'unknown')
        paddle_report['paddleocr_version'] = importlib.metadata.version('paddleocr')
        paddle_report['paddlex_version'] = importlib.metadata.version('paddlex')
        paddle_report['cuda_enabled'] = bool(paddle.device.is_compiled_with_cuda())
        paddle_report['gpu_visible_to_paddle'] = bool(
            paddle.device.cuda.device_count() if paddle_report['cuda_enabled'] else 0
        )
        if not paddle_report['cuda_enabled']:
            paddle_report['runtime_note'] = (
                'Colab exposed a GPU, but the installed Paddle wheel is CPU-only; '
                'this is not a GPU performance measurement.'
            )

        started = time.perf_counter()
        pipeline = PaddleOCRVL()
        predictions = list(pipeline.predict(input=str(sample[0])))
        paddle_report['latency_ms'] = round((time.perf_counter() - started) * 1000, 2)
        paddle_report['prediction_count'] = len(predictions)
        paddle_report['prediction_types'] = [type(item).__name__ for item in predictions]
        paddle_report['status'] = 'success'
    except Exception as error:
        paddle_report['status'] = 'runtime_failed'
        paddle_report['error_type'] = type(error).__name__
        paddle_report['error'] = str(error)
        paddle_report['traceback_tail'] = traceback.format_exc()[-5000:]
paddle_path = RUN_ROOT / 'paddleocr-vl-smoke-report.json'
paddle_path.write_text(json.dumps(paddle_report, indent=2) + '\n')
print('PaddleOCR-VL status:', paddle_report['status'])
print('PaddleOCR-VL report:', paddle_path)
versions = {
    key: paddle_report.get(key)
    for key in ('paddleocr_version', 'paddle_version', 'paddlex_version')
}
print('Paddle versions:', versions)
print('Paddle CUDA enabled:', paddle_report.get('cuda_enabled'))
if paddle_report.get('runtime_note'):
    print('Runtime note:', paddle_report['runtime_note'])
if paddle_report.get('error'):
    print('Error:', paddle_report['error'])

## 6. Export a private benchmark artifact

Gold transcriptions stay private. The next cell creates a deterministic 50-page review package with provisional routing buckets, source hashes, OCR drafts, and blank annotation fields. Inspect each referenced PDF page privately, then fill the gold fields before mapping reviewed rows into `BenchmarkCase` records.

In [ ]:
report = {
    'evidence_scope': 'colab_smoke',
    'native_results': native_results,
    'docling_result': docling_result,
    'notes': 'Smoke run only; not a quality claim.',
}
report_path = RUN_ROOT / 'ocr-smoke-report.json'
report_path.write_text(json.dumps(report, indent=2) + '\n')
print(report_path)

## 7. Build the private gold-review package

This writes only to Google Drive. Provisional buckets are heuristics based on native text length and must be confirmed by a human reviewer; they are not quality labels.

In [ ]:
from collections import Counter

review_root = DATA_ROOT / 'gold-review'
review_root.mkdir(parents=True, exist_ok=True)

# Re-filter here so the cell is safe even if a fresh run includes
# metadata files or macOS sidecar files in the source snapshot.
def valid_pdfs(paths):
    return [
        path
        for path in sorted(paths)
        if not path.name.startswith('._') and path.open('rb').read(5) == b'%PDF-'
    ]

candidates = []
for source_id, paths in (
    ('awesome-pitch-decks', valid_pdfs(awesome_pdfs)),
    ('pitch-deckz', valid_pdfs(pitch_deckz_pdfs)),
):
    for path in paths:
        try:
            reader = PdfReader(str(path))
            digest = sha256_file(path)
        except Exception as error:
            print('skip:', path.name, type(error).__name__, str(error))
            continue
        for page_number, page in enumerate(reader.pages, start=1):
            draft_text = (page.extract_text() or '').strip()
            candidates.append({
                'source_id': source_id,
                'relative_path': str(path.relative_to(DATA_ROOT)),
                'source_file': path.name,
                'document_sha256': digest,
                'page_number': page_number,
                'native_characters': len(draft_text),
                'native_draft_text': draft_text,
            })

bucket_specs = (
    ('born_digital', lambda row: row['native_characters'] >= 200, 20),
    ('scanned', lambda row: row['native_characters'] == 0, 15),
    ('chart_table', lambda row: 40 <= row['native_characters'] < 200, 10),
    ('difficult', lambda row: 0 < row['native_characters'] < 40, 5),
)
selected = []
selected_keys = set()
for bucket, predicate, target in bucket_specs:
    eligible = [row for row in candidates if predicate(row)]
    eligible.sort(key=lambda row: hashlib.sha256(
        f"{row['document_sha256']}:{row['page_number']}".encode()
    ).hexdigest())
    for row in eligible:
        key = (row['document_sha256'], row['page_number'])
        if key in selected_keys:
            continue
        row = dict(row)
        row['case_id'] = (
            f"{row['source_id']}:{row['document_sha256'][:12]}:"
            f"p{row['page_number']:03d}"
        )
        row['provisional_bucket'] = bucket
        row['review_status'] = 'needs_review'
        row['gold_text'] = ''
        row['gold_numbers'] = []
        row['gold_headings'] = []
        row['gold_bullets'] = []
        row['gold_tables'] = []
        selected.append(row)
        selected_keys.add(key)
        if sum(1 for item in selected if item['provisional_bucket'] == bucket) >= target:
            break

selected.sort(key=lambda row: (row['source_id'], row['source_file'], row['page_number']))
counts = Counter(row['provisional_bucket'] for row in selected)
review_manifest = {
    'schema_version': 1,
    'evidence_scope': 'private_gold_review_v1',
    'sample_target': {bucket: target for bucket, _, target in bucket_specs},
    'sampled_pages': len(selected),
    'bucket_counts': dict(sorted(counts.items())),
    'notes': [
        'Buckets are provisional routing hints, not quality labels.',
        'Replace blank gold fields only after inspecting the corresponding PDF page.',
        'Keep this directory private; never commit PDFs, drafts, or gold annotations.',
    ],
    'records': selected,
}
canonical = json.dumps(review_manifest, sort_keys=True, separators=(',', ':'), ensure_ascii=False)
review_manifest['manifest_sha256'] = hashlib.sha256(canonical.encode('utf-8')).hexdigest()
manifest_path = review_root / 'gold-review-manifest.json'
manifest_path.write_text(json.dumps(review_manifest, indent=2, ensure_ascii=False) + '\n')
records_path = review_root / 'gold-review-records.jsonl'
records_path.write_text(''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in selected))
instructions = review_root / 'REVIEW_INSTRUCTIONS.txt'
instructions.write_text(
    'Private gold review\n\n'
    'For each record, open the referenced PDF and page number. Confirm the\n'
    'provisional bucket, then enter an exact English transcription in gold_text.\n'
    'Fill gold_numbers, gold_headings, gold_bullets, and gold_tables only when\n'
    'the page contains those structures. Set review_status to reviewed.\n\n'
    'Do not treat native_draft_text as ground truth. Keep all files in Drive.\n'
)
print('candidate pages scanned:', len(candidates))
print('selected pages:', len(selected))
print('bucket counts:', dict(sorted(counts.items())))
print('manifest:', manifest_path)
print('editable records:', records_path)
print('instructions:', instructions)

## 8. Generate private machine-assisted drafts

This optional cell runs Docling page-by-page on the selected sample and writes machine drafts beside the review records. Machine output is never copied into `gold_text`: every row remains `machine_draft_unverified` until a human checks the corresponding PDF page. Empty drafts are valid failure cases and should be routed to a second OCR engine or manually transcribed.

In [ ]:
import importlib.metadata as metadata
import json
import time

review_root = DATA_ROOT / 'gold-review'
records_path = review_root / 'gold-review-records.jsonl'
records = [json.loads(line) for line in records_path.read_text().splitlines() if line.strip()]
docling_version = metadata.version('docling')
machine_records = []
status_counts = Counter()
started_all = time.perf_counter()

for index, original in enumerate(records, start=1):
    row = dict(original)
    page_started = time.perf_counter()
    draft_text = ''
    draft_status = 'failed'
    error = None
    try:
        source_path = DATA_ROOT / row['relative_path']
        result = converter.convert(
            str(source_path),
            page_range=(row['page_number'], row['page_number']),
            raises_on_error=False,
        )
        draft_text = result.document.export_to_markdown().strip()
        draft_status = 'drafted' if draft_text else 'empty'
    except Exception as exc:
        error = f'{type(exc).__name__}: {exc}'
    row['machine_draft_text'] = draft_text
    row['machine_draft_engine'] = 'docling'
    row['machine_draft_engine_version'] = docling_version
    row['machine_draft_status'] = draft_status
    row['machine_draft_latency_ms'] = round((time.perf_counter() - page_started) * 1000, 2)
    if error:
        row['machine_draft_error'] = error
    # Keep all gold fields untouched; machine output is not ground truth.
    row['review_status'] = 'machine_draft_unverified'
    machine_records.append(row)
    status_counts[draft_status] += 1
    print(
        f"{index}/{len(records)} {row['source_file']} "
        f"p{row['page_number']}: {draft_status}, "
        f"chars={len(draft_text)}, ms={row['machine_draft_latency_ms']}"
    )

machine_path = review_root / 'gold-review-records-with-machine-drafts.jsonl'
machine_path.write_text(
    ''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in machine_records)
)
manifest_path = review_root / 'gold-review-manifest.json'
manifest = json.loads(manifest_path.read_text())
manifest.update({
    'draft_engine': 'docling',
    'draft_engine_version': docling_version,
    'drafted_records_path': str(machine_path),
    'drafted_pages': len(machine_records),
    'draft_status_counts': dict(status_counts),
    'gold_text_policy': 'blank_until_human_review',
    'review_status_policy': 'machine_draft_unverified',
})
# Recompute the manifest hash after adding draft metadata.
manifest.pop('manifest_sha256', None)
canonical = json.dumps(manifest, sort_keys=True, separators=(',', ':'), ensure_ascii=False)
manifest['manifest_sha256'] = hashlib.sha256(canonical.encode('utf-8')).hexdigest()
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False) + '\n')

report = {
    'evidence_scope': 'private_50_page_machine_draft',
    'engine': 'docling',
    'engine_version': docling_version,
    'pages_attempted': len(machine_records),
    'status_counts': dict(status_counts),
    'total_latency_ms': round((time.perf_counter() - started_all) * 1000, 2),
    'records_path': str(machine_path),
    'gold_text_policy': 'blank_until_human_review',
}
report_path = RUN_ROOT / 'machine-draft-report.json'
report_path.write_text(json.dumps(report, indent=2) + '\n')
print('machine drafts:', machine_path)
print('report:', report_path)
print('status counts:', dict(status_counts))
print('total latency ms:', report['total_latency_ms'])